# Análisis unidimensional

Todas las estadísticas usan las semanas 2010–2021 (historia y entrenamiento); la prueba sigue sellada.

In [1]:
import os  # Variables de entorno y rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable1 (paquete e1)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import contextlib, io  # Silenciar una salida de diagnóstico
from joblib.externals.loky import cpu_count  # Conteo de núcleos de joblib
with contextlib.redirect_stderr(io.StringIO()):  # En Windows 11 joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché) sin ensuciar las salidas
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Figuras con varios paneles
from src import config as cf  # Configuración del proyecto (horizonte, periodos, variables)
from e1 import datos as d, estadistica as es, espacial as sp, modelo_base as mb  # Código del entregable
from e1.figuras import mostrar, mapa_base, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO, PERIODO_COLOR, CUADRANTE_COLOR  # Estilo
np.random.seed(cf.SEMILLA)  # Semilla global (42)
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 40)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
from scipy import stats  # Pruebas de normalidad
df = d.panel()  # Panel completo
eda = d.entrenamiento_eda(df)  # Semanas t de 2010–2021
mod = d.modelado(df)  # Filas aptas para modelar
tr, va, te = d.particion(mod)  # Entrenamiento (para las variables derivadas)
print(f"Filas del EDA: {len(eda):,}")  # Tamaño

Filas del EDA: 235,376


## Variables numéricas originales: estadísticos, forma y normalidad

Las pruebas de normalidad (D'Agostino–Pearson y Jarque–Bera) se aplican a una muestra de 5 000 filas: con cientos de miles de filas rechazarían cualquier desviación mínima, así que se leen junto con la asimetría y la curtosis.

In [2]:
num = ["casos_totales", "casos_0_4", "casos_5_14", "casos_15_64", "casos_65_plus", "casos_m", "casos_f",  # Conteos
       "temp_mean", "prec_total", "ndvi_mean", "oni_anom", "oni_total", "Flujo_in", "poblacion"]  # Continuas
muestra = eda[num].sample(5000, random_state=cf.SEMILLA)  # Muestra para las pruebas
filas = []  # Resultados
for v in num:  # Cada variable
    x = eda[v]  # Serie
    q1, q3 = x.quantile([0.25, 0.75])  # Cuartiles
    iqr = q3 - q1  # Rango intercuartílico
    filas.append({"variable": v, "media": x.mean(), "mediana": x.median(), "desv. est.": x.std(),  # Centro y dispersión
                  "p1": x.quantile(0.01), "p5": x.quantile(0.05), "p25": q1, "p75": q3, "p95": x.quantile(0.95), "p99": x.quantile(0.99),  # Percentiles
                  "asimetría": x.skew(), "curtosis": x.kurt(),  # Forma
                  "atípicos IQR (%)": ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).mean() * 100,  # Regla de Tukey
                  "p normalidad (K²)": stats.normaltest(muestra[v])[1], "p Jarque-Bera": stats.jarque_bera(muestra[v])[1]})  # Normalidad
uni = pd.DataFrame(filas).set_index("variable")  # Tabla
uni.round(3)

,media,mediana,desv. est.,p1,p5,p25,p75,p95,p99,asimetría,curtosis,atípicos IQR (%),p normalidad (K²),p Jarque-Bera
variable,,,,,,,,,,,,,,
casos_totales,3.5550,0.0000,20.2740,0.0000,0.0000,0.0000,2.0000,12.0000,62.0000,18.2710,473.3340,10.0980,0.0000,0.0000
casos_0_4,0.3430,0.0000,2.0030,0.0000,0.0000,0.0000,0.0000,1.0000,7.0000,15.2840,359.0420,11.6100,0.0000,0.0000
casos_5_14,1.1510,0.0000,6.5530,0.0000,0.0000,0.0000,0.0000,4.0000,22.0000,16.3320,406.3000,22.6940,0.0000,0.0000
casos_15_64,1.8960,0.0000,12.0440,0.0000,0.0000,0.0000,1.0000,6.0000,32.0000,22.2920,686.6410,11.8360,0.0000,0.0000
casos_65_plus,0.1640,0.0000,1.0680,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,19.6710,594.3870,7.7290,0.0000,0.0000
casos_m,1.8670,0.0000,10.2470,0.0000,0.0000,0.0000,1.0000,7.0000,33.0000,17.4740,438.2050,12.1590,0.0000,0.0000
casos_f,1.6870,0.0000,10.1520,0.0000,0.0000,0.0000,1.0000,6.0000,30.0000,18.9700,509.5220,10.6310,0.0000,0.0000
temp_mean,23.1830,22.4630,3.0150,13.2500,18.2040,22.2330,24.2800,28.6500,30.2330,-0.2160,2.1400,18.1570,0.0000,0.0000
prec_total,42.5320,25.1000,51.8550,0.0000,0.0000,4.1000,62.0000,145.0000,232.0000,2.2590,8.3740,4.6380,0.0000,0.0000


In [3]:
def histograma(x, bins=60, log=False):  # Histograma precalculado (la figura no embebe todos los datos)
    x = np.log1p(x) if log else x  # Escala logarítmica opcional
    conteo, bordes = np.histogram(x, bins=bins)  # Conteos
    return (bordes[:-1] + bordes[1:]) / 2, conteo / conteo.sum()  # Centros y proporciones

paneles = [("casos_totales", True), ("poblacion", True), ("Flujo_in", True), ("temp_mean", False),  # (variable, ¿log?)
           ("prec_total", False), ("ndvi_mean", False), ("oni_anom", False), ("oni_total", False)]
fig = make_subplots(2, 4, subplot_titles=[f"log(1 + {v})" if lg else v for v, lg in paneles])  # Rejilla
for i, (v, lg) in enumerate(paneles):  # Cada variable
    xc, pr = histograma(eda[v], log=lg)  # Histograma
    fig.add_bar(x=xc, y=pr, marker_color=AZUL, showlegend=False, row=i // 4 + 1, col=i % 4 + 1)  # Barras
fig.update_layout(title="Distribuciones de las variables numéricas originales (2010–2021)", bargap=0)  # Formato
mostrar(fig, 520)  # Muestra

In [4]:
fig = go.Figure()  # Diagramas de caja precalculados
for v in ["temp_mean", "prec_total", "ndvi_mean", "oni_anom"]:  # Variables en escala comparable tras estandarizar
    x = (eda[v] - eda[v].mean()) / eda[v].std()  # Estandariza para un mismo eje
    q1, me, q3 = x.quantile([0.25, 0.5, 0.75])  # Cuartiles
    iqr = q3 - q1  # Rango intercuartílico
    fig.add_box(name=v, q1=[q1], median=[me], q3=[q3], lowerfence=[max(x.min(), q1 - 1.5 * iqr)],  # Caja de Tukey
                upperfence=[min(x.max(), q3 + 1.5 * iqr)], marker_color=AZUL, showlegend=False)
    fig.add_scatter(x=[v, v], y=[x.min(), x.max()], mode="markers", marker=dict(color=ROJO, size=7, symbol="x"), showlegend=False)  # Extremos
fig.update_layout(title="Diagramas de caja estandarizados (x: mínimo y máximo)", yaxis_title="desviaciones estándar")  # Formato
mostrar(fig, 380)  # Muestra

**Interpretación.**

* **Casos.** Distribución de conteo extremadamente asimétrica (asimetría > 15, curtosis en cientos): la mediana es 0 y la media unos pocos casos. Es la firma de un fenómeno epidémico con muchas semanas tranquilas y pocas explosivas. Necesita $\log(1+x)$; los grupos de edad y sexo tienen la misma forma y son colineales con el total (suman exactamente el total).
* **Población y flujo.** Colas largas por las grandes ciudades: también se usan en escala logarítmica.
* **Temperatura.** Bimodal por la imputación (un pico artificial en 22,4 °C y la temperatura real de tierra caliente alrededor de 26–28 °C).
* **Precipitación.** Asimétrica a la derecha, con semanas secas frecuentes y lluvias extremas en el Pacífico.
* **NDVI.** Ligeramente asimétrico a la izquierda (zonas urbanas y secas con menos vegetación).
* **ONI.** Una sola serie nacional repetida en todos los municipios: su variabilidad es solo temporal.
* **Normalidad.** Ninguna variable es normal (todas las pruebas rechazan aun con 5 000 filas). No es un problema para la regresión logística, que no supone normalidad de las predictoras, pero sí justifica las transformaciones y el uso de pruebas no paramétricas en el análisis bidimensional.

## Variables predictoras derivadas (entrenamiento)

Las 34 predictoras se describen en el capítulo de preprocesamiento; aquí se resume su distribución.

In [5]:
der = tr[cf.FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T  # Estadísticos
der["asimetría"] = tr[cf.FEATURES].skew()  # Forma
der["curtosis"] = tr[cf.FEATURES].kurt()  # Colas
der["únicos"] = tr[cf.FEATURES].nunique()  # Cardinalidad
der.round(3)

,count,mean,std,min,5%,50%,95%,max,asimetría,curtosis,únicos
lc_l0,"132,728.0000",0.5570,0.9130,0.0000,0.0000,0.0000,2.4850,6.7810,2.0780,5.0160,356
lc_l1,"132,728.0000",0.5570,0.9140,0.0000,0.0000,0.0000,2.4850,6.7810,2.0760,4.9990,355
lc_l2,"132,728.0000",0.5580,0.9140,0.0000,0.0000,0.0000,2.4850,6.7810,2.0720,4.9750,354
lc_l3,"132,728.0000",0.5590,0.9150,0.0000,0.0000,0.0000,2.4850,6.7810,2.0700,4.9560,353
lc_m4,"132,728.0000",0.5580,0.8580,0.0000,0.0000,0.1730,2.3840,6.7100,2.3500,6.6100,8006
lc_m8,"132,728.0000",0.5590,0.8430,0.0000,0.0000,0.1730,2.3510,6.6080,2.4080,6.9660,12945
lc_m12,"132,728.0000",0.5600,0.8330,0.0000,0.0000,0.2070,2.3220,6.4860,2.4290,7.1220,16220
brote_l0,"132,728.0000",0.1030,0.3040,0.0000,0.0000,0.0000,1.0000,1.0000,2.6100,4.8110,2
brote_m8,"132,728.0000",0.1040,0.2390,0.0000,0.0000,0.0000,0.7500,1.0000,2.5800,5.7530,9
ratio_umbral,"132,728.0000",-0.2810,0.8150,-5.5650,-1.6090,0.0000,1.0990,3.8290,-0.0040,1.2330,3437


**Interpretación.** Tras el logaritmo, la historia de casos (`lc_*`) sigue siendo asimétrica porque la mayoría de las semanas tienen 0 casos, pero las colas se reducen de cientos a unas pocas unidades de curtosis. La incidencia (`incid`) conserva una cola larga (municipios pequeños con un brote tienen incidencias altísimas); el estandarizado del Pipeline no la elimina, y por eso el modelo también recibe la versión logarítmica de los casos.

## Variables categóricas y de calendario

In [6]:
eda = eda.assign(fase_enso=pd.cut(eda["oni_anom"], [-9, -0.5, 0.5, 9], labels=["La Niña", "Neutral", "El Niño"]))  # Fase ENSO
cat = {"región": eda.groupby("region")["COD_MUN_N"].nunique(),  # Municipios por región
       "departamento": eda.groupby("departamento")["COD_MUN_N"].nunique().sort_values(),  # Municipios por departamento
       "fase ENSO (semanas)": eda.drop_duplicates("time_idx")["fase_enso"].value_counts(),  # Semanas por fase
       "temperatura imputada (filas)": eda["temp_mean_missing"].value_counts()}  # Bandera
card = pd.DataFrame({"cardinalidad": {"MES": eda["MES"].nunique(), "SEMANA": eda["SEMANA"].nunique(), "región": eda["region"].nunique(),  # Cardinalidad
                                      "departamento": eda["departamento"].nunique(), "fase ENSO": 3, "temp_mean_missing": 2}})
card

,cardinalidad
MES,12
SEMANA,53
departamento,32
fase ENSO,3
región,5
temp_mean_missing,2


In [7]:
dep = cat["departamento"]  # Municipios por departamento
fig = make_subplots(1, 3, column_widths=[0.5, 0.25, 0.25], subplot_titles=("Municipios por departamento", "Municipios por región", "Semanas por fase ENSO"))  # Tres paneles
fig.add_bar(x=dep.values, y=dep.index, orientation="h", marker_color=np.where(dep.values < 3, ROJO, AZUL), showlegend=False, row=1, col=1)  # Departamentos (rojo: raros)
fig.add_bar(x=cat["región"].index, y=cat["región"].values, marker_color=VERDE, showlegend=False, row=1, col=2)  # Regiones
fe = cat["fase ENSO (semanas)"]  # Fases
fig.add_bar(x=fe.index.astype(str), y=fe.values, marker_color=[AZUL if f == "La Niña" else GRIS if f == "Neutral" else ROJO for f in fe.index], showlegend=False, row=1, col=3)  # Fases
fig.update_layout(title="Frecuencias de las variables categóricas (rojo: departamentos con menos de 3 municipios)")  # Título
mostrar(fig, 620)  # Muestra
semanas = eda.drop_duplicates("time_idx")["SEMANA"].value_counts().sort_index()  # Semanas por número de semana
print(f"Semana 53: {semanas.get(53, 0)} ocurrencias (años 2015 y 2020) frente a {semanas.drop(53).min()}–{semanas.drop(53).max()} del resto")  # Categoría rara
print("Departamentos con menos de 3 municipios en el panel:", ", ".join(dep[dep < 3].index))  # Raros

Semana 53: 2 ocurrencias (años 2015 y 2020) frente a 12–12 del resto
Departamentos con menos de 3 municipios en el panel: Guainía, Quindío, Vaupés, Bogotá, D.C., Guaviare, Amazonas, Arauca


**Interpretación y tratamiento de categorías raras.**

* **Departamento** (32 categorías) tiene varias categorías raras (departamentos amazónicos y San Andrés con 1–2 municipios). Codificarlo como variable indicadora haría que el modelo memorice lugares y no generalice a zonas nuevas; **no se usa como predictora**. Se usa como bloque de la validación espacial y para agregar la región.
* **Región** (5 categorías) no tiene categorías raras; se usa en el EDA para describir la heterogeneidad espacial.
* **Semana 53** solo ocurre en 2015 y 2020. Se codifica con seno y coseno de $2\pi\,\text{semana}/52$, de modo que la semana 53 queda junto a la semana 1, sin crear una categoría aparte.
* **Fase ENSO**: de 626 semanas, 287 son neutrales, 179 de La Niña y 160 de El Niño; ninguna fase es rara. Se conserva el ONI continuo (no las fases) para no perder información.
* **Temperatura imputada**: 64 % de las filas; es la bandera del mecanismo de faltantes y se conserva.